# Sudan Climate Data Analysis (2015–2026)

**Data Source:** NASA POWER climate reanalysis (January 2015 – March 2026)

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)
COUNTRY = 'Sudan'
CSV_PATH = os.path.join(DATA_DIR, 'sudan.csv')
CLEAN_PATH = os.path.join(DATA_DIR, 'sudan_clean.csv')
print(f"✓ {COUNTRY}")

In [ ]:
df = pd.read_csv(CSV_PATH)
df['Country'] = COUNTRY
df['DATE'] = pd.to_datetime(df['YEAR'].astype(str) + '-' + df['DOY'].astype(str).str.zfill(3), format='%Y-%j', errors='coerce')
df['Year'] = df['DATE'].dt.year
df['Month'] = df['DATE'].dt.month
print(f"Shape: {df.shape}, Range: {df['DATE'].min()} to {df['DATE'].max()}")

In [ ]:
df = df.replace(-999.0, np.nan)
df = df.drop_duplicates().reset_index(drop=True)
threshold = int(df.shape[1] * 0.7)
df = df.dropna(thresh=threshold).reset_index(drop=True)
weather_vars = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']
df[weather_vars] = df[weather_vars].fillna(method='ffill').fillna(method='bfill')
print(f"✓ Cleaned: {len(df)} rows")

In [ ]:
print(df[['T2M', 'PRECTOTCORR', 'RH2M']].describe().round(2))

In [ ]:
df['YearMonth'] = df['DATE'].dt.to_period('M')
monthly = df.groupby('YearMonth').agg({'T2M': 'mean', 'PRECTOTCORR': 'sum'}).reset_index()
monthly['YearMonth'] = monthly['YearMonth'].dt.to_timestamp()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8))
ax1.plot(monthly['YearMonth'], monthly['T2M'], linewidth=2, color='red')
ax1.set_title(f'{COUNTRY} — Monthly Avg T2M', fontweight='bold')
ax1.set_ylabel('Temperature (°C)')
ax1.grid(True, alpha=0.3)

ax2.bar(monthly['YearMonth'], monthly['PRECTOTCORR'], alpha=0.7, width=20)
ax2.set_title(f'{COUNTRY} — Monthly Precipitation', fontweight='bold')
ax2.set_ylabel('Precipitation (mm)')
ax2.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
print(f"Extreme heat days (T2M_MAX>35°C): {(df['T2M_MAX']>35).sum()}")
print(f"Dry days (<1mm): {(df['PRECTOTCORR']<1).sum()} ({(df['PRECTOTCORR']<1).sum()/len(df)*100:.1f}%)")
df.to_csv(CLEAN_PATH, index=False)
print(f"✓ Exported: {CLEAN_PATH}")